In [ ]:
import re
import pandas as pd
import nltk

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC

from nltk.sentiment import SentimentIntensityAnalyzer
from transformers import pipeline

# Download NLTK resources
nltk.download('vader_lexicon')
nltk.download('punkt')

[nltk_data] Downloading package vader_lexicon to /root/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [10]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

max_chunk_length = 3000
text_chunk = article_text[:max_chunk_length]

# Initialize the tokenizer and model
model_name = "facebook/bart-large-cnn"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Tokenize and generate summary
inputs = tokenizer([text_chunk], max_length=1024, return_tensors="pt", truncation=True)
summary_ids = model.generate(inputs["input_ids"], max_length=150, min_length=50, do_sample=False)
summary_text = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print("--- Article Summary ---")
print(summary_text)

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

--- Article Summary ---
OpenAI CEO Sam Altman said his company is experiencing "unbelievable" growth. Altman was speaking at the TED 2025 conference in Vancouver. The company recently closed a $40 billion funding round, valuing it at $300 billion.


In [11]:
# Identify Key Topics using Zero-Shot Classification
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")
candidate_labels = ["artificial intelligence", "business growth", "ethics and safety", "future predictions", "company politics"]

topics_result = classifier(text_chunk, candidate_labels)
print("\n--- Key Topics ---")
for label, score in zip(topics_result['labels'], topics_result['scores']):
    print(f"{label}: {score:.4f}")

# LLM Sentiment
sentiment_pipeline = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")
llm_sentiment = sentiment_pipeline(text_chunk[:512]) # Truncate for model limit
print(f"\n--- LLM Sentiment ---")
print(llm_sentiment)

# Traditional Sentiment (VADER)
sia = SentimentIntensityAnalyzer()
vader_sentiment = sia.polarity_scores(article_text)
print(f"\n--- Traditional Sentiment (VADER) ---")
print(vader_sentiment)

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]


--- Key Topics ---
business growth: 0.3374
ethics and safety: 0.2624
future predictions: 0.1582
artificial intelligence: 0.1256
company politics: 0.1165


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]


--- LLM Sentiment ---
[{'label': 'POSITIVE', 'score': 0.9985820055007935}]

--- Traditional Sentiment (VADER) ---
{'neg': 0.047, 'neu': 0.807, 'pos': 0.146, 'compound': 0.9993}


In [12]:
emotion_labels = ["tense", "optimistic", "defensive", "excited", "anxious", "neutral"]
emotion_result = classifier(text_chunk, emotion_labels)

print("--- General Emotion ---")
print(f"Dominant Emotion: {emotion_result['labels'][0]} ({emotion_result['scores'][0]:.4f})")
for label, score in zip(emotion_result['labels'][1:3], emotion_result['scores'][1:3]):
    print(f"Secondary: {label} ({score:.4f})")

--- General Emotion ---
Dominant Emotion: excited (0.3262)
Secondary: tense (0.2632)
Secondary: anxious (0.1942)


In [13]:
theme_labels = [
    "The rapid and unprecedented growth of AI and OpenAI.",
    "The ethical dilemmas and safety concerns surrounding AGI.",
    "Leadership challenges and internal conflicts at major tech companies.",
    "The economic impact of AI on global markets."
]

theme_result = classifier(text_chunk, theme_labels)

print("--- Main Theme ---")
print(theme_result['labels'][0])

--- Main Theme ---
The rapid and unprecedented growth of AI and OpenAI.
